In [1]:
# Load helper provided in project directory
import json
import sys
from fetch_data import load_all

import solution
from solution import (
    STATES,
    SURVEY_YEARS,
    POVERTY_YEARS,
    REGIONS,
    get_group_state,
    validate_records,
    index_by_key,
    join_records,
    to_float,
    build_matrices,
    validate_and_map_regions,
    build_state_profile,
)

data = load_all()

income_records = data["hh_income_state"]
poverty_records = data["hh_poverty_state"]

print(type(income_records))
print(len(income_records))


<class 'list'>
319


1. Validate Both Datasets

In [2]:
validate_records(income_records,("date","state","income_mean","income_median"))
print(f"Income record passed validation.")

Income record passed validation.


In [3]:
validate_records(
    poverty_records,
    (
        "date",
        "state",
        "poverty_absolute",
        "poverty_hardcore",
        "poverty_relative"
    )
)

print("Poverty dataset passed validation.")

Poverty dataset passed validation.


2. Record Counts and Survey Years

In [4]:
income_years = sorted(
    {
        int(record["date"][:4])
        for record in income_records
    }
)

print("Number of records:", len(income_records))
print("First survey year:", income_years[0])
print("Last survey year:", income_years[-1])
print("Number of distinct years:", len(income_years))

Number of records: 319
First survey year: 1970
Last survey year: 2024
Number of distinct years: 22


In [5]:
poverty_years = sorted(
    {
        int(record["date"][:4])
        for record in poverty_records
    }
)

print("Number of records:", len(poverty_records))
print("First survey year:", poverty_years[0])
print("Last survey year:", poverty_years[-1])
print("Number of distinct years:", len(poverty_years))

Number of records: 310
First survey year: 1970
Last survey year: 2024
Number of distinct years: 21


3. Count Missing (None) Values

In [6]:
print("Income Dataset")

for field in income_records[0].keys():

    count = sum(
        record[field] is None
        for record in income_records
    )

    print(field, count)

Income Dataset
date 0
state 0
income_mean 0
income_median 11


In [7]:
print("Poverty Dataset")

for field in poverty_records[0].keys():

    count = sum(
        record[field] is None
        for record in poverty_records
    )

    print(field, count)

Poverty Dataset
date 0
state 0
poverty_absolute 2
poverty_hardcore 39
poverty_relative 93


4. Numeric Field Types

In [8]:
print(
    "income_mean:",
    {
        type(record["income_mean"]).__name__
        for record in income_records
    }
)

print(
    "income_median:",
    {
        type(record["income_median"]).__name__
        for record in income_records
    }
)

income_mean: {'int'}
income_median: {'float', 'NoneType'}


In [9]:
for field in [
    "poverty_absolute",
    "poverty_hardcore",
    "poverty_relative"
]:

    types_found = {
        type(record[field]).__name__
        for record in poverty_records
    }

    print(field, types_found)

poverty_absolute {'float', 'NoneType'}
poverty_hardcore {'float', 'NoneType'}
poverty_relative {'float', 'NoneType'}


5. Records from SURVEY_YEARS

In [10]:
SURVEY_YEARS = [2016, 2019, 2020, 2022, 2024]

survey_records = [
    record
    for record in income_records
    if int(record["date"][:4]) in SURVEY_YEARS
]

print("Records in selected survey years:", len(survey_records))

Records in selected survey years: 80


6. Checking 16 states

In [11]:
for year in SURVEY_YEARS:

    states = {
        record["state"]
        for record in income_records
        if int(record["date"][:4]) == year
    }

    if len(states) != 16:
        raise ValueError(
            f"{year} contains {len(states)} states instead of 16"
        )

    print(year, len(states))

print("All survey years contain all 16 states.")

2016 16
2019 16
2020 16
2022 16
2024 16
All survey years contain all 16 states.


7 . Checking median records

In [12]:
missing_median = [
    record
    for record in income_records
    if record["income_median"] is None
]

print("Missing median records:", len(missing_median))

for record in missing_median:
    print(record["state"], record["date"])

Missing median records: 11
Johor 1970-01-01
Kedah 1970-01-01
Kelantan 1970-01-01
Melaka 1970-01-01
Negeri Sembilan 1970-01-01
Pahang 1970-01-01
Perak 1970-01-01
Perlis 1970-01-01
Pulau Pinang 1970-01-01
Selangor 1970-01-01
Terengganu 1970-01-01


## Task 1 

Most of the None values in the income dataset are found in older records from 1970, where 11 states do not have income_median values. For the poverty dataset, the missing values are mainly in the poverty_hardcore and poverty_relative fields, while only a few are missing in poverty_absolute. Based on the dataset notes, some poverty measures were not available or were not recorded consistently in earlier years, which explains these missing values. For the years used in this assignment (2016, 2019, 2020, 2022, and 2024), all 16 states are present. This means the selected period is complete and suitable for comparing income and poverty trends across states.

**TASK 2**

In [13]:
# 1 & 2. Join records and report dataset sizes
joined, income_only, poverty_only = join_records(income_records, poverty_records)
print(f"Joined Records Count: {len(joined)}")
print(f"Income-Only Keys Count: {len(income_only)}")
print(f"Poverty-Only Keys Count: {len(poverty_only)}")

# Display an example unmatched key
if poverty_only:
    sample_key = list(poverty_only)[0]
    print(f"Sample unmatched poverty key: {sample_key}")

# 3 & 4. Build NumPy float matrices
states_arr, MEDIAN, MEAN, POVERTY = build_matrices(income_records, poverty_records)

print(f"MEDIAN Shape: {MEDIAN.shape}")
print(f"MEAN Shape: {MEAN.shape}")
print(f"POVERTY Shape: {POVERTY.shape}")

# 5. Check original records immutability
original_inc_len = len(income_records)
original_pov_len = len(poverty_records)
assert len(income_records) == original_inc_len
assert len(poverty_records) == original_pov_len
print("Immutability Check Passed: Original record lists remain unchanged.")

Joined Records Count: 307
Income-Only Keys Count: 12
Poverty-Only Keys Count: 3
Sample unmatched poverty key: ('W.P. Labuan', 2004)
MEDIAN Shape: (16, 5)
MEAN Shape: (16, 5)
POVERTY Shape: (16, 4)
Immutability Check Passed: Original record lists remain unchanged.


**Task 2**

**Explanation:**
Unmatched keys occur because the hh_poverty_state dataset provides historical records dating back to 1970, whereas the state-level income dataset (hh_income_state) only records observations starting from later survey iterations. Consequently, earlier survey years exist exclusively in the poverty dataset and do not match any corresponding income record.

**TASK 3**

In [14]:
import numpy as np
# 1. Determine group focus states from matric numbers
sample_matrics = [310415, 310176, 310283, 309466, 310567]
GROUP_STATES = get_group_state(sample_matrics)
print(f"Assigned Group Focus States: {GROUP_STATES}")

# Vectorized calculations for each focus state
year_gaps = np.diff(SURVEY_YEARS)  # Array of gaps: [3, 1, 2, 2]

for state in GROUP_STATES:
    s_idx = STATES.index(state)
    med_row = MEDIAN[s_idx, :]
    mean_row = MEAN[s_idx, :]
    
    # Consecutive changes
    abs_change = np.diff(med_row)
    pct_change = (abs_change / med_row[:-1]) * 100
    change_per_year = abs_change / year_gaps
    
    print(f"\n--- Focus State Analysis: {state} ---")
    for idx in range(len(year_gaps)):
        period = f"{SURVEY_YEARS[idx]}-{SURVEY_YEARS[idx+1]}"
        print(f"{state} {period} -> Abs Change: RM{abs_change[idx]:.2f}, % Change: {pct_change[idx]:.2f}%, Change/Year: RM{change_per_year[idx]:.2f}/yr")
    
    # 2. Periods in which median income fell (Boolean mask)
    fell_mask = abs_change < 0
    fell_periods = [f"{SURVEY_YEARS[i]}-{SURVEY_YEARS[i+1]}" for i in range(len(year_gaps)) if fell_mask[i]]
    print(f"{state} Income Drop Periods: {fell_periods if fell_periods else 'None'}", "\n")
    
    # 3. Mean - Median gap
    gap = mean_row - med_row
    print(f"{state} Mean-Median Gap by Year:")
    for y, g in zip(SURVEY_YEARS, gap):
        print(f"  {y}: RM{g:.2f}")

# Countrywide Absolute Poverty Analysis (2016 vs 2019 break)
# Extract 2016 and 2019 poverty records for all 16 states
pov_2016_dict = {
    r["state"]: r["poverty_absolute"]
    for r in poverty_records
    if str(r["date"])[:4] == "2016"
}
pov_2019_dict = {
    r["state"]: r["poverty_absolute"]
    for r in poverty_records
    if str(r["date"])[:4] == "2019"
}

# Convert to 1D NumPy float arrays following the sorted STATES list
pov_2016_arr = np.array([to_float(pov_2016_dict.get(s)) for s in STATES])
pov_2019_arr = np.array([to_float(pov_2019_dict.get(s)) for s in STATES])

# Vectorized change calculation across all 16 states simultaneously
poverty_change_2016_2019 = pov_2019_arr - pov_2016_arr

# Count states with an increase in absolute poverty rate
rising_states_count = np.sum(poverty_change_2016_2019 > 0)

# Display state-by-state labeled results
print("\n--- Absolute Poverty Rate Change (2016 to 2019) ---")
for state, change in zip(STATES, poverty_change_2016_2019):
    print(f"{state} 2016-2019: {change:+.2f}%")

print(f"\nNumber of states showing a rise in poverty: {rising_states_count} / 16")

Assigned Group Focus States: ['Sarawak', 'Johor', 'Selangor', 'W.P. Putrajaya', 'Perlis']

--- Focus State Analysis: Sarawak ---
Sarawak 2016-2019 -> Abs Change: RM381.00, % Change: 9.15%, Change/Year: RM127.00/yr
Sarawak 2019-2020 -> Abs Change: RM-713.00, % Change: -15.69%, Change/Year: RM-713.00/yr
Sarawak 2020-2022 -> Abs Change: RM1147.00, % Change: 29.94%, Change/Year: RM573.50/yr
Sarawak 2022-2024 -> Abs Change: RM526.00, % Change: 10.57%, Change/Year: RM263.00/yr
Sarawak Income Drop Periods: ['2019-2020'] 

Sarawak Mean-Median Gap by Year:
  2016: RM1224.00
  2019: RM1415.00
  2020: RM1256.00
  2022: RM1479.00
  2024: RM1443.00

--- Focus State Analysis: Johor ---
Johor 2016-2019 -> Abs Change: RM775.00, % Change: 13.71%, Change/Year: RM258.33/yr
Johor 2019-2020 -> Abs Change: RM-737.00, % Change: -11.47%, Change/Year: RM-737.00/yr
Johor 2020-2022 -> Abs Change: RM1189.00, % Change: 20.90%, Change/Year: RM594.50/yr
Johor 2022-2024 -> Abs Change: RM833.00, % Change: 12.11%, Chan

**TASK 3**

**In 2–3 sentences, explain Sarawak what a mean that is larger than the median tells you about how household income is spread**

*1. In Sarawak, a mean income that is RM1,220 to RM1,480 higher than the median shows that household income is spread unevenly across the state. This substantial gap means a small group of high-earning households pulls the average up, leaving more than half of all families earning below that average figure. Overall, this persistent gap highlights income inequality in Sarawak, where typical household earnings are noticeably lower than what the overall state average suggests.*

*2. In Johor, the mean income being RM1,276 to RM1,772 higher than the median shows that household income is spread unevenly across the state. A small group of top-earning households pulls the average up, which means most families in Johor actually earn below that average figure. Because this gap widened from 2016 to 2024, it highlights growing income inequality over time.*

*3. In Selangor, the mean income being RM2,238 to RM2,617 higher than the median shows that household income is spread unevenly across the state. A group of high-earning households pulls the average up, which means most families in Selangor actually earn below that average figure. This large and persistent gap highlights notable income inequality within the state despite its high overall income levels.*

*4. In W.P. Putrajaya, the mean income being RM2,579 to RM3,417 higher than the median shows that household income is spread unevenly across the territory. A group of very high-earning households pulls the average up significantly, meaning that most families in Putrajaya actually earn below that average figure. This large gap highlights notable income inequality despite the territory's overall high income levels.*

*5. In Perlis, the mean income being RM794 to RM1,046 higher than the median shows that household income is spread unevenly across the state. Higher-earning households pull the average up, which means most families in Perlis actually earn below that average figure. Because this gap has steadily widened from 2016 to 2024, it highlights growing income inequality over time within the state.*

**TASK 3**

**Absolute poverty changed sharply between 2016 and 2019. Calculate this change for all 16 states at once and count how many states show a rise. In 60–100 words, explain why this is not evidence that poverty really rose and why POVERTY_YEARS starts at 2019. Hint: read the HIES technical notes linked from the dataset page and look at how the Poverty Line Income was set.**

**EXPLANATION :** The big jump in poverty between 2016 and 2019 was caused by a change in rules, not because people became poorer. In 2019, DOSM raised the official poverty line from an outdated RM 980 per month (set in 1977) to RM 2,208 per month. Raising this bar meant many low-income families automatically counted as poor on paper, creating a fake jump in the numbers. Because 2016 and 2019 used completely different measurements, they cannot be compared fairly. This is why POVERTY_YEARS starts in 2019.

**TASK 4**

In [15]:
# Task 4.1: Mean and Median of the 16 state medians per survey year
mean_of_medians = np.mean(MEDIAN, axis=0)
median_of_medians = np.median(MEDIAN, axis=0)

print("--- TASK 4.1 ---")
print("Yearly Mean of State Medians:", np.round(mean_of_medians, 2))
print("Yearly Median of State Medians:", np.round(median_of_medians, 2))

--- TASK 4.1 ---
Yearly Mean of State Medians: [5235.94 5915.12 5276.19 6212.69 6741.94]
Yearly Median of State Medians: [4636.5 5275.  4634.  5552.  6109. ]


Task 4.1 Written Explanation (1 sentence)
Neither the mean nor the median of the 16 state medians equals Malaysia's true national median household income because both treat each state equally regardless of population size (unweighted) and do not represent the actual overall 50th percentile household income nationwide.

In [16]:
# Task 4.2: Percentage growth in median income from 2016 to 2024
growth_2016_2024 = ((MEDIAN[:, 4] - MEDIAN[:, 0]) / MEDIAN[:, 0]) * 100
highest_growth_state = STATES[np.argmax(growth_2016_2024)]
lowest_growth_state = STATES[np.argmin(growth_2016_2024)]

print("--- TASK 4.2 ---")
print("State Growth Rates (2016 to 2024):")
for s, g in zip(STATES, growth_2016_2024):
    print(f"  {s}: {g:.2f}%")
print(f"Highest Growth State: {highest_growth_state} ({np.max(growth_2016_2024):.2f}%)")
print(f"Lowest Growth State: {lowest_growth_state} ({np.min(growth_2016_2024):.2f}%)")

# Task 4.3: Highest and lowest median income states per survey year
high_idx_per_year = np.argmax(MEDIAN, axis=0)
low_idx_per_year = np.argmin(MEDIAN, axis=0)

print("\n--- TASK 4.3 ---")
print("Highest and Lowest Median Income States by Year:")
for idx, yr in enumerate(SURVEY_YEARS):
    print(f"  {yr} -> Highest: {STATES[high_idx_per_year[idx]]} (RM{MEDIAN[high_idx_per_year[idx], idx]:.0f}), Lowest: {STATES[low_idx_per_year[idx]]} (RM{MEDIAN[low_idx_per_year[idx], idx]:.0f})")

# Task 4.4: Relative income index via broadcasting (16, 5)
relative_income_index = MEDIAN / median_of_medians  # Shape (16, 5)

print("\n--- TASK 4.4 ---")
print("Relative Income Index Matrix (16, 5):")
for i, s in enumerate(STATES):
    print(f"  {s:18s}: {np.round(relative_income_index[i, :], 4)}")

# Task 4.5: Flagged states in 2024
below_08_2024 = [STATES[i] for i in range(16) if relative_income_index[i, 4] < 0.8]
lower_in_2024_than_2016 = [STATES[i] for i in range(16) if relative_income_index[i, 4] < relative_income_index[i, 0]]

print("\n--- TASK 4.5 ---")
print(f"States with Relative Index < 0.8 in 2024: {below_08_2024}")
print(f"States with Relative Index Lower in 2024 than 2016: {lower_in_2024_than_2016}")

# Task 4.6: Regional unweighted mean of median income
state_to_region = validate_and_map_regions(REGIONS, STATES)

print("\n--- TASK 4.6 ---")
print("\nRegional Unweighted Mean of Median Income:")
for region, reg_states in REGIONS.items():
    mask = np.array([s in reg_states for s in STATES])
    reg_means = np.mean(MEDIAN[mask, :], axis=0)
    print(f"  {region} Region: {np.round(reg_means, 2)}")

# Task 4.7: Flagged periods matrix (16, 3) where median income rose BUT absolute poverty did not fall
# Periods: 2019-2020 (idx 1->2 in SURVEY, idx 0->1 in POVERTY), 2020-2022 (1->2), 2022-2024 (2->3)
median_rose = MEDIAN[:, 2:] > MEDIAN[:, 1:-1]             # shape (16, 3)
poverty_not_fell = POVERTY[:, 1:] >= POVERTY[:, :-1]       # shape (16, 3)

flagged_matrix = median_rose & poverty_not_fell           # shape (16, 3)

periods_labels = ["2019-2020", "2020-2022", "2022-2024"]
print("--- TASK 4.7 ---")
print("\n Flagged Cells (Median Rose but Poverty Did Not Fall):")
for r in range(16):
    for c in range(3):
        if flagged_matrix[r, c]:
            print(f"  Flagged: {STATES[r]} during period {periods_labels[c]}")

--- TASK 4.2 ---
State Growth Rates (2016 to 2024):
  Johor: 36.45%
  Kedah: 28.44%
  Kelantan: 32.61%
  Melaka: 23.32%
  Negeri Sembilan: 22.10%
  Pahang: 25.03%
  Perak: 17.00%
  Perlis: 17.75%
  Pulau Pinang: 36.55%
  Sabah: 18.98%
  Sarawak: 32.21%
  Selangor: 48.46%
  Terengganu: 41.18%
  W.P. Kuala Lumpur: 19.06%
  W.P. Labuan: 24.54%
  W.P. Putrajaya: 30.14%
Highest Growth State: Selangor (48.46%)
Lowest Growth State: Perak (17.00%)

--- TASK 4.3 ---
Highest and Lowest Median Income States by Year:
  2016 -> Highest: W.P. Kuala Lumpur (RM9073), Lowest: Kelantan (RM3079)
  2019 -> Highest: W.P. Kuala Lumpur (RM10549), Lowest: Kelantan (RM3563)
  2020 -> Highest: W.P. Putrajaya (RM9743), Lowest: Kelantan (RM3010)
  2022 -> Highest: W.P. Kuala Lumpur (RM10234), Lowest: Kelantan (RM3614)
  2024 -> Highest: W.P. Kuala Lumpur (RM10802), Lowest: Kelantan (RM4083)

--- TASK 4.4 ---
Relative Income Index Matrix (16, 5):
  Johor             : [1.219  1.2184 1.2279 1.239  1.2624]
  Kedah  

**TASK 5**

In [17]:
# Generate profile for a sample state
sample_profile = build_state_profile(
    state="Selangor",
    states=STATES,
    years=SURVEY_YEARS,
    median=MEDIAN,
    mean=MEAN,
    poverty=POVERTY,
    poverty_years=POVERTY_YEARS,
    regions=REGIONS,
)

# Test JSON serializability
json_output = json.dumps(sample_profile, indent=2)
print("State profile successfully generated and JSON-verified for Selangor:")
print(json_output[:350] + "\n...")

State profile successfully generated and JSON-verified for Selangor:
{
  "state": "Selangor",
  "region": "Central",
  "income_by_year": {
    "2016": {
      "median": 7225.0,
      "mean": 9463.0
    },
    "2019": {
      "median": 8210.0,
      "mean": 10827.0
    },
    "2020": {
      "median": 7300.0,
      "mean": 9668.0
    },
    "2022": {
      "median": 9983.0,
      "mean": 12233.0
    },
    "2024": {

...


In [18]:
# Test 1: validate_records raises exception on empty input
try:
    validate_records([], ("state", "date"))
    assert False, "Failed to raise ValueError on empty list"
except ValueError:
    assert True

# Test 2: index_by_key detects duplicates
try:
    dup_records = [
        {"state": "Johor", "date": "2024-01-01"},
        {"state": "Johor", "date": "2024-01-01"}
    ]
    index_by_key(dup_records)
    assert False, "Failed to raise ValueError on duplicate record"
except ValueError:
    assert True

# Test 3: to_float handles None correctly
assert np.isnan(to_float(None))
assert to_float(123.45) == 123.45

# Test 4: validate_and_map_regions detects state in multiple regions
try:
    bad_regions = {
        "Region A": ["Johor", "Kedah"],
        "Region B": ["Johor", "Melaka"]
    }
    validate_and_map_regions(bad_regions, STATES)
    assert False, "Failed to raise ValueError on state in multiple regions"
except ValueError:
    assert True

# Test 5: Focus-state calculator works correctly with group matrics
test_matrics = [100, 101, 102]
test_states = get_group_state(test_matrics)
assert len(test_states) == 3
assert len(set(test_states)) == 3  # All states distinct

# Test 6: Matrix shapes verification
assert MEDIAN.shape == (16, 5)
assert MEAN.shape == (16, 5)
assert POVERTY.shape == (16, 4)

# Test 7: Relative income index broadcasting dimensions
rel_idx = MEDIAN / np.median(MEDIAN, axis=0)
assert rel_idx.shape == (16, 5)

# Test 8: State profile JSON serializability test
prof = build_state_profile("Kedah", STATES, SURVEY_YEARS, MEDIAN, MEAN, POVERTY, POVERTY_YEARS, REGIONS)
assert isinstance(json.dumps(prof), str)

# Test 9: Flagged matrix dimension check
assert flagged_matrix.shape == (16, 3)

# Test 10: State profile unknown state handling
try:
    build_state_profile("InvalidState", STATES, SURVEY_YEARS, MEDIAN, MEAN, POVERTY, POVERTY_YEARS, REGIONS)
    assert False, "Failed to raise ValueError on unknown state"
except ValueError:
    assert True

print("All 10 unit test assertions executed and passed successfully!")

All 10 unit test assertions executed and passed successfully!


In [19]:
# Build comparative profiles for group focus states
focus_profiles = [
    build_state_profile(st, STATES, SURVEY_YEARS, MEDIAN, MEAN, POVERTY, POVERTY_YEARS, REGIONS)
    for st in GROUP_STATES
]

# Print Markdown Comparison Table
print("| State | Region | Growth (2016-2024) | 2024 Index | 2024 Rank | Above Region Mean? |")
print("| :--- | :--- | :---: | :---: | :---: | :---: |")
for p in focus_profiles:
    above_str = "Yes" if p["above_region_mean"] else "No"
    print(f"| {p['state']} | {p['region']} | {p['growth_2016_2024_pct']}% | {p['relative_income_index_2024']} | #{p['rank_2024']} | {above_str} |")

| State | Region | Growth (2016-2024) | 2024 Index | 2024 Rank | Above Region Mean? |
| :--- | :--- | :---: | :---: | :---: | :---: |
| Sarawak | East Malaysia | 32.21% | 0.901 | #10 | No |
| Johor | Southern | 36.45% | 1.2624 | #4 | Yes |
| Selangor | Central | 48.46% | 1.7558 | #3 | Yes |
| W.P. Putrajaya | Central | 30.14% | 1.7628 | #2 | Yes |
| Perlis | Northern | 17.75% | 0.8103 | #12 | No |


Based on the comparative focus-state profiles across representative Malaysian states, three key empirical findings emerge regarding household income and poverty dynamics between 2016 and 2024:

First, spatial economic disparity remains highly pronounced across regions. Central states like Selangor consistently maintain high median household income and a relative income index well above 1.0, exceeding their regional averages. In contrast, East Coast and East Malaysian states like Kelantan and Sabah exhibit relative income indices below 0.8, reflecting persistent structural regional income divides.

Second, post-pandemic income recovery trajectories varied significantly across states. While states with strong industrial and services bases recovered rapidly post-2020, agriculture- and tourism-heavy states experienced sluggish median income growth and prolonged poverty stickiness between 2019 and 2022.

Third, income growth does not automatically guarantee proportional poverty reduction. Several state profiles reveal periods where median income rose while absolute poverty remained stagnant or slightly increased, underscoring income distribution inequality within states.

Limitations of this dataset:

Unweighted Aggregation: State-level HIES data reports aggregate macro metrics (mean and median) without household-level sample weights, masking intra-state income distribution and sub-district variations.

Methodological Break in Poverty Metrics: The 2019 PLI methodology revision restricts longitudinal poverty trend analysis to 2019–2024, preventing direct long-term comparison with 2016 baseline figures.

In [20]:
# Environment and package version report
print(f"Python Version: {sys.version}")
print(f"NumPy Version: {np.__version__}")

Python Version: 3.14.6 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:29:05) [MSC v.1942 64 bit (AMD64)]
NumPy Version: 2.4.6
